# N18 — Radio Transcription (Whisper ASR)

This notebook transcribes all team radio audio files using OpenAI Whisper. The output is a single CSV (`radios_raw.csv`) with one row per audio file, consumed by N17 for labeling and N19 for VADER analysis.

## Step 0 — Setup

In [1]:
# ── Step 0 · Setup ────────────────────────────────────────────────────────────
import sys
import os
import glob
import warnings
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import whisper
import librosa
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

repo_root = Path.cwd()
while not (repo_root / ".git").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# ── Paths ──────────────────────────────────────────────────────────────────────
AUDIO_DIR = repo_root / "data" / "raw" / "radio_audio"
PROC_DIR  = repo_root / "data" / "processed" / "radio_nlp"
OUTPUTS   = repo_root / "notebooks" / "nlp" / "outputs"

for d in [AUDIO_DIR, PROC_DIR, OUTPUTS]:
    d.mkdir(parents=True, exist_ok=True)

print(f"repo_root : {repo_root}")
print(f"AUDIO_DIR : {AUDIO_DIR}")
print(f"PROC_DIR  : {PROC_DIR}")
print(f"OUTPUTS   : {OUTPUTS}")
print(f"CUDA available: {torch.cuda.is_available()}")

repo_root : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager
AUDIO_DIR : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\raw\radio_audio
PROC_DIR  : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\processed\radio_nlp
OUTPUTS   : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\notebooks\nlp\outputs
CUDA available: True


## Loading audio files

---

All audio files are scanned from `data/raw/radio_audio/driver_*/` and collected as a list of metadata dicts before transcription.

In [2]:
# Default audio dir — override with AUDIO_DIR from Step 0
def load_audio_files(audio_dir=str(AUDIO_DIR)):
    """
    Load all audio files from the directory structure.
    
    Parameters:
    -----------
    audio_dir : str
        Path to the main audio directory that contains driver subdirectories
        
    Returns:
    --------
    list
        List of dictionaries containing audio file paths and metadata
    """
    audio_files = []

    # Find all driver directories. For this, we use driver_* for matching every directory
    driver_dirs = glob.glob(os.path.join(audio_dir, 'driver_*'))

    for driver_dir in driver_dirs:
        # Extract the driver number out of the directory name
        driver_num = os.path.basename(driver_dir).replace('driver_(', '').replace(',)', '')
        # Find all audio files in this driver directory.
        # For now, only mp3 is in our directory. 
        # However, we add all those extensions if the directory changes in the future
        files = glob.glob(os.path.join(driver_dir, '*.mp3')) + \
                glob.glob(os.path.join(driver_dir, '*.wav')) + \
                glob.glob(os.path.join(driver_dir, '*.m4a')) + \
                glob.glob(os.path.join(driver_dir, '*.ogg'))
        
        for file_path in files:
            # Get the filename without the extension
            filename = os.path.basename(file_path)

            # Add to the list of audio files
            audio_files.append(
                {
                    "driver": driver_num,
                    "file_path": file_path,
                    "filename": filename
                }
            )
    print(f"Found {len(audio_files)} audio files across {len(driver_dirs)} driver directories")

    return audio_files

In [3]:
def preview_audio_files(audio_files, n=5):
    if audio_files:
        print(f"\nFirst {n} audio files:")
        for f in audio_files[:n]:
            print(f"  Driver: {f['driver']}, File: {f['filename']}")
    else:
        print("No audio files found. Check AUDIO_DIR path.")


audio_files = load_audio_files()
preview_audio_files(audio_files)

Found 659 audio files across 22 driver directories

First 5 audio files:
  Driver: driver_1, File: driver_1_belgium_radio_39.mp3
  Driver: driver_1, File: driver_1_belgium_radio_40.mp3
  Driver: driver_1, File: driver_1_belgium_radio_60.mp3
  Driver: driver_1, File: driver_1_belgium_radio_62.mp3
  Driver: driver_1, File: driver_1_belgium_radio_63.mp3


## Transcribing the audios with Whisper

Whisper `turbo` is used for transcription — it runs on GPU when available (detected in Step 0). The transcription pipeline is split into three focused functions: model loading, single-file transcription, and CSV export.

In [4]:
def load_whisper_model(model_name="turbo"):
    print(f"Loading Whisper '{model_name}' model...")
    return whisper.load_model(model_name)


def transcribe_single_file(model, audio_file):
    file_path = os.path.normpath(audio_file['file_path'])
    audio, sr = librosa.load(file_path, sr=16000, mono=True)
    duration = librosa.get_duration(y=audio, sr=sr)
    result = model.transcribe(audio, task="transcribe", language="en", fp16=torch.cuda.is_available())
    full_text = " ".join(seg["text"].strip() for seg in result["segments"])
    return {
        'driver':    audio_file['driver'],
        'filename':  audio_file['filename'],
        'file_path': file_path,
        'text':      full_text,
        'duration':  duration,
    }


def save_transcriptions(results, output_csv):
    df = pd.DataFrame(results)
    output_path = Path(output_csv)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f"Saved {len(df)} transcriptions to {output_path}")
    return df


def transcribe_audio_files(audio_files, model_name="turbo", output_csv=str(PROC_DIR / "radios_raw.csv")):
    model = load_whisper_model(model_name)
    results = []
    print(f"Transcribing {len(audio_files)} audio files...")
    for i, audio_file in enumerate(audio_files):
        try:
            print(f"  [{i+1}/{len(audio_files)}] {audio_file['filename']}")
            entry = transcribe_single_file(model, audio_file)
            results.append(entry)
            print(f"  → {entry['text']}")
        except Exception as e:
            print(f"  Error processing {audio_file['filename']}: {e}")
    return save_transcriptions(results, output_csv)

### Transcribing all audios

Before running, confirm GPU is available — transcription is significantly slower on CPU.

In [5]:
def check_gpu():
    if torch.cuda.is_available():
        print(f"GPU available: {torch.cuda.get_device_name(0)}")
        print(f"CUDA version: {torch.version.cuda}")
    else:
        print("No GPU detected. Using CPU (will be much slower).")


check_gpu()

GPU available: NVIDIA GeForce RTX 5070 Laptop GPU
CUDA version: 12.8


In [6]:
full_df = transcribe_audio_files(
    audio_files,
    model_name="turbo",
    output_csv=str(PROC_DIR / "radios_raw.csv")
)

print(f"\nTranscription complete — {len(full_df)} files processed.")

Loading Whisper 'turbo' model...


100%|█████████████████████████████████████| 1.51G/1.51G [00:44<00:00, 36.8MiB/s]


Transcribing 659 audio files...
  [1/659] driver_1_belgium_radio_39.mp3
  → So don't forget Max, use your head please. Are we both doing it or what? You just follow my instruction. No, I want to know if both cars do it. Max, please follow my instruction and trust it. Thank you.
  [2/659] driver_1_belgium_radio_40.mp3
  → Okay Max, we're expecting rain in about 9 or 10 minutes. What are your thoughts? That you can get there or should we box? We'd need to box this lap to cover Leclerc. I can't see the weather, can I? I don't know.
  [3/659] driver_1_belgium_radio_60.mp3
  → $%&&S& $%&!
  [4/659] driver_1_belgium_radio_62.mp3
  → You might find this lap that you meet a little bit more water.
  [5/659] driver_1_belgium_radio_63.mp3
  → Just another two or three minutes to get through this.
  [6/659] driver_1_belgium_radio_68.mp3
  → So settle into standard race management now Max.
  [7/659] driver_1_belgium_radio_81.mp3
  → Use a lot of attire on an Outland Max. Not sure that was sensible.

## Conclusions

This notebook produces `data/processed/radio_nlp/radios_raw.csv` with the following schema:

| Column | Description |
|--------|-------------|
| `driver` | Driver number extracted from the directory name |
| `filename` | Original audio filename |
| `file_path` | Absolute path to the audio file |
| `text` | Full Whisper transcription (all segments joined) |
| `duration` | Audio duration in seconds |

Next step: **N17** (`N17_radio_labeling.ipynb`) — filter post-race messages and manually label sentiment.